# **Nested CV–Optimized Traditional ML Models — Final Retraining and External Validation**

This notebook rebuilds and applies the best traditional ML models for each AD-relevant target using pre-selected hyperparameters obtained from Bayesian optimization within nested cross-validation.

For each target, the workflow:

- Loads training and independent external test set with corresponding RFECV reduced feature ssets.

- Reconstructs each estimator as an sklearn.pipeline.Pipeline (with scaling where appropriate) and sets the target-specific best hyperparameters.

- Retrains models on the full training set using these selected(optimized) settings (no test information is used during training).

- Generates predictions for both training and external test sets, saving per-target prediction files and a combined all-target external test predictions table.


- Model selection and ranking were performed using the outer-loop RMSE from nested cross-validation to provide an unbiased estimate of generalization performance and to avoid information leakage. Ensemble evaluation is conducted exclusively on the external test set, using the saved per-model predictions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd
import numpy as np

from collections import OrderedDict

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor



In [ ]:
reu_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/descriptors/reduced_features_reu_2"

out_dir  = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions"
model_dir = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/models/saved_tradML_models"
os.makedirs(out_dir, exist_ok=True)

meta_cols = ['molecule_chembl_id', 'canonical_smiles', 'activity_class', 'pIC50']
targets = ["5-HT6","ache", "bace1", "buche","esr1","gsk-3beta", "mao-b"]



### Best params into one dictionary


In [ ]:
best_params = {
    "5-HT6": {
        "RandomForest": {
            "model__max_depth": 18,
            "model__min_samples_split": 2,
            "model__n_estimators": 500
        },
        "XGBoost": {
            "model__colsample_bytree": 1.0,
            "model__learning_rate": 0.025887784600853534,
            "model__max_depth": 7,
            "model__n_estimators": 271,
            "model__subsample": 0.5
        },
        "GradientBoosting": {
            "model__learning_rate": 0.08073994859329678,
            "model__max_depth": 3,
            "model__n_estimators": 201
        },
        "SVR": {
            "model__C": 1.319003730278866,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 5
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 3.7634674415015654,
            "model__hidden_layer_sizes": 75,
            "model__learning_rate_init": 5.671542189490181e-05,
            "model__solver": "sgd"
        }
    },

    "ache": {
        "RandomForest": {
            "model__max_depth": 36,
            "model__min_samples_split": 2,
            "model__n_estimators": 500
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5,
            "model__learning_rate": 0.07741903729403656,
            "model__max_depth": 7,
            "model__n_estimators": 300,
            "model__subsample": 1.0
        },
        "GradientBoosting": {
            "model__learning_rate": 0.0767808050856075,
            "model__max_depth": 7,
            "model__n_estimators": 292
        },
        "SVR": {
            "model__C": 3.0551341830311385,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 3
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 1.5247791391944723,
            "model__hidden_layer_sizes": 96,
            "model__learning_rate_init": 0.06381101112960001,
            "model__solver": "sgd"
        }
    },

    "bace1": {
        "RandomForest": {
            "model__max_depth": 33,
            "model__min_samples_split": 16,
            "model__n_estimators": 212
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5271645889894799,
            "model__learning_rate": 0.03765961219616658,
            "model__max_depth": 8,
            "model__n_estimators": 244,
            "model__subsample": 1.0
        },
        "GradientBoosting": {
            "model__learning_rate": 0.04922408286051131,
            "model__max_depth": 6,
            "model__n_estimators": 300
        },
        "SVR": {
            "model__C": 1.6606492208851877,
            "model__epsilon": 0.20552446343146835,
            "model__gamma": 0.001258815937361985,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 8
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 0.08023246075377977,
            "model__hidden_layer_sizes": 181,
            "model__learning_rate_init": 0.0004605580210141211,
            "model__solver": "adam"
        }
    },

    "buche": {
        "RandomForest": {
            "model__max_depth": 23,
            "model__min_samples_split": 2,
            "model__n_estimators": 415
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5,
            "model__learning_rate": 0.04703395065480419,
            "model__max_depth": 10,
            "model__n_estimators": 249,
            "model__subsample": 0.7469755057542441
        },
        "GradientBoosting": {
            "model__learning_rate": 0.060963619408668866,
            "model__max_depth": 6,
            "model__n_estimators": 300
        },
        "SVR": {
            "model__C": 3.3190939761653477,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 3
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 0.08023246075377977,
            "model__hidden_layer_sizes": 181,
            "model__learning_rate_init": 0.0004605580210141211,
            "model__solver": "adam"
        }
    },

    "esr1": {
        "RandomForest": {
            "model__max_depth": 11,
            "model__min_samples_split": 2,
            "model__n_estimators": 499
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5014308669733738,
            "model__learning_rate": 0.03174431677644471,
            "model__max_depth": 7,
            "model__n_estimators": 242,
            "model__subsample": 0.5
        },
        "GradientBoosting": {
            "model__learning_rate": 0.09309792968967405,
            "model__max_depth": 3,
            "model__n_estimators": 300
        },
        "SVR": {
            "model__C": 1.4887739517349046,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 6
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 1.5247791391944723,
            "model__hidden_layer_sizes": 96,
            "model__learning_rate_init": 0.06381101112960001,
            "model__solver": "sgd"
        }
    },

    "gsk-3beta": {
        "RandomForest": {
            "model__max_depth": 36,
            "model__min_samples_split": 2,
            "model__n_estimators": 500
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5,
            "model__learning_rate": 0.02978246296926844,
            "model__max_depth": 7,
            "model__n_estimators": 300,
            "model__subsample": 1.0
        },
        "GradientBoosting": {
            "model__learning_rate": 0.061309677716235376,
            "model__max_depth": 7,
            "model__n_estimators": 251
        },
        "SVR": {
            "model__C": 2.125804531901401,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 4
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 0.08023246075377977,
            "model__hidden_layer_sizes": 181,
            "model__learning_rate_init": 0.0004605580210141211,
            "model__solver": "adam"
        }
    },

    "mao-b": {
        "RandomForest": {
            "model__max_depth": 50,
            "model__min_samples_split": 3,
            "model__n_estimators": 488
        },
        "XGBoost": {
            "model__colsample_bytree": 0.5,
            "model__learning_rate": 0.06748116613898338,
            "model__max_depth": 7,
            "model__n_estimators": 300,
            "model__subsample": 1.0
        },
        "GradientBoosting": {
            "model__learning_rate": 0.05115273709641036,
            "model__max_depth": 6,
            "model__n_estimators": 300
        },
        "SVR": {
            "model__C": 3.876182231690694,
            "model__kernel": "rbf"
        },
        "KNeighborsRegressor": {
            "model__n_neighbors": 3
        },
        "MLPRegressor": {
            "model__activation": "logistic",
            "model__alpha": 1.5247791391944723,
            "model__hidden_layer_sizes": 96,
            "model__learning_rate_init": 0.06381101112960001,
            "model__solver": "sgd"
        }
    }
}


## Build Pipeline, Apply params

In [ ]:
# make_model() — this function returns a sklearn.pipeline.Pipeline

def make_model(model_name: str):
    model_name = model_name.strip()

    if model_name == "RandomForest":
        model = RandomForestRegressor(random_state=42)
        pipe = Pipeline([("model", model)])
        return pipe

    if model_name == "GradientBoosting":
        model = GradientBoostingRegressor(random_state=42)
        pipe = Pipeline([("model", model)])
        return pipe

    if model_name == "XGBoost":
        if not HAS_XGB:
            raise RuntimeError("XGBoost not installed/importable in this environment.")
        # initialize the model with safe base settings. The optimized hyperparameters will replace them
        model = XGBRegressor(
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )
        pipe = Pipeline([("model", model)])
        return pipe

    if model_name == "SVR":
        model = SVR()
        pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
        return pipe

    if model_name == "KNeighborsRegressor":
        model = KNeighborsRegressor()
        pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])  # KNN sensitive to scale
        return pipe

    if model_name == "MLPRegressor":
        model = MLPRegressor(
            random_state=42,
            max_iter=2000,
            early_stopping=True
        )
        pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
        return pipe

    raise ValueError(f"Unknown model: {model_name}")


In [ ]:
def train_and_save_target(target):
    df_train = pd.read_csv(f"{reu_path}/{target}_train_reu.csv").dropna(how="any")
    X_train = df_train.drop(columns=meta_cols)
    y_train = df_train["pIC50"].values

    # Save feature column order so you can align new data later
    feat_fp = os.path.join(model_dir, f"{target}_feature_columns.joblib")
    joblib.dump(list(X_train.columns), feat_fp)

    for model_name, params in best_params[target].items():
        pipe = make_model(model_name)
        pipe.set_params(**params)
        pipe.fit(X_train, y_train)

        out_fp = os.path.join(model_dir, f"{target}__{model_name}.joblib")
        joblib.dump(pipe, out_fp)
        print("Saved:", out_fp)

## Train + predict + save (per target + master file)

In [ ]:
def load_target_data(target):
    train_fp = f"{reu_path}/{target}_train_reu.csv"
    test_fp  = f"{reu_path}/{target}_test_reu.csv"

    df_train = pd.read_csv(train_fp).dropna(how="any")
    df_test  = pd.read_csv(test_fp).dropna(how="any")

    X_train = df_train.drop(columns=meta_cols)
    y_train = df_train["pIC50"].values

    X_test = df_test.drop(columns=meta_cols)
    y_test = df_test["pIC50"].values

    return df_train, df_test, X_train, y_train, X_test, y_test


In [ ]:
all_test_rows = []

for target in targets:
    if target not in best_params:
        print(f"[SKIP] No best_params found for target: {target}")
        continue

    df_train, df_test, X_train, y_train, X_test, y_test = load_target_data(target)

    # Store predictions in a single dataframe per target
    test_out = df_test[["molecule_chembl_id", "canonical_smiles", "pIC50"]].copy()
    train_out = df_train[["molecule_chembl_id", "canonical_smiles", "pIC50"]].copy()

    for model_name, params in best_params[target].items():
        try:
            pipe = make_model(model_name)
            pipe.set_params(**params)
            pipe.fit(X_train, y_train)

            test_out[f"pred_{model_name}"] = pipe.predict(X_test)
            train_out[f"pred_{model_name}"] = pipe.predict(X_train)

            print(f"[OK] {target} - {model_name}")

        except Exception as e:
            print(f"[FAIL] {target} - {model_name}: {e}")

    # Save per-target preds
    test_csv = f"{out_dir}/{target}_test_preds_tradML.csv"
    train_csv = f"{out_dir}/{target}_train_preds_tradML.csv"
    test_out.to_csv(test_csv, index=False)
    train_out.to_csv(train_csv, index=False)

    #  append to master
    tmp = test_out.copy()
    tmp.insert(0, "target", target)
    all_test_rows.append(tmp)

# Save master file
if all_test_rows:
    master = pd.concat(all_test_rows, ignore_index=True)
    master_fp = f"{out_dir}/all_targets_test_preds_tradML.csv"
    master.to_csv(master_fp, index=False)
    print("Saved:", master_fp)


[OK] 5-HT6 - RandomForest
[OK] 5-HT6 - XGBoost
[OK] 5-HT6 - GradientBoosting
[OK] 5-HT6 - SVR
[OK] 5-HT6 - KNeighborsRegressor
[OK] 5-HT6 - MLPRegressor
[OK] ache - RandomForest
[OK] ache - XGBoost
[OK] ache - GradientBoosting
[OK] ache - SVR
[OK] ache - KNeighborsRegressor
[OK] ache - MLPRegressor
[OK] bace1 - RandomForest
[OK] bace1 - XGBoost
[OK] bace1 - GradientBoosting
[OK] bace1 - SVR
[OK] bace1 - KNeighborsRegressor
[OK] bace1 - MLPRegressor
[OK] buche - RandomForest
[OK] buche - XGBoost
[OK] buche - GradientBoosting
[OK] buche - SVR
[OK] buche - KNeighborsRegressor
[OK] buche - MLPRegressor
[OK] esr1 - RandomForest
[OK] esr1 - XGBoost
[OK] esr1 - GradientBoosting
[OK] esr1 - SVR
[OK] esr1 - KNeighborsRegressor
[OK] esr1 - MLPRegressor
[OK] gsk-3beta - RandomForest
[OK] gsk-3beta - XGBoost
[OK] gsk-3beta - GradientBoosting
[OK] gsk-3beta - SVR
[OK] gsk-3beta - KNeighborsRegressor
[OK] gsk-3beta - MLPRegressor
[OK] mao-b - RandomForest
[OK] mao-b - XGBoost
[OK] mao-b - GradientBo